# Seleksi, Augmentasi, dan Penggabungan Data Historis UD Sentosa

Notebook ini membuat `data_historis_gabungan.csv` dengan aturan label baru. Label akhir tidak langsung mengikuti label historis atau label vehicle saja, tetapi dihitung dari kombinasi jumlah komponen KM historis yang memenuhi kondisi servis dan label `Need_Maintenance` dari `vehicle_maintenance_data.csv`.

## 1. Import Library dan Konfigurasi

In [1]:
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
TARGET_LAYAK = 3000
TARGET_PERLU_SERVIS = 10000
BATAS_KM_SERVIS = 10000

DATA_HISTORIS_PATH = Path("data_historis.csv")
VEHICLE_DATA_PATH = Path("vehicle_maintenance_data.csv")
OUTPUT_PATH = Path("data_historis_gabungan.csv")

## 2. Load Dataset

`data_historis.csv` digunakan sebagai sumber atribut historis kendaraan UD Sentosa. `vehicle_maintenance_data.csv` digunakan sebagai sumber atribut kondisi kendaraan dan label `Need_Maintenance`.

In [2]:
data_historis = pd.read_csv(DATA_HISTORIS_PATH)
vehicle_data = pd.read_csv(VEHICLE_DATA_PATH)

print(f"Jumlah data_historis: {len(data_historis)} baris")
print(f"Jumlah vehicle_data: {len(vehicle_data)} baris")
display(data_historis.head())
display(vehicle_data.head())

Jumlah data_historis: 200 baris
Jumlah vehicle_data: 50000 baris


,id,umur_kendaraan,jarak_tempuh_tahun,frekuensi_servis_tahun,km_oli,km_rem,km_busi,km_ban,kelas,created_at,updated_at
0,806,2,8730,3,1137,6419,5367,10811,Layak,2026-07-02 02:26:05,2026-07-02 02:26:05
1,807,3,9460,4,1274,6838,5734,11622,Layak,2026-07-02 02:26:05,2026-07-02 02:26:05
2,808,4,10190,5,1411,7257,6101,12433,Layak,2026-07-02 02:26:05,2026-07-02 02:26:05
3,809,1,10920,6,1548,7676,6468,13244,Layak,2026-07-02 02:26:05,2026-07-02 02:26:05
4,810,2,11650,2,1685,8095,6835,14055,Layak,2026-07-02 02:26:05,2026-07-02 02:26:05


,Vehicle_Model,Mileage,Maintenance_History,Reported_Issues,Vehicle_Age,Fuel_Type,Transmission_Type,Engine_Size,Odometer_Reading,Last_Service_Date,Warranty_Expiry_Date,Owner_Type,Insurance_Premium,Service_History,Accident_History,Fuel_Efficiency,Tire_Condition,Brake_Condition,Battery_Status,Need_Maintenance
0,Truck,58765,Good,0,4,Electric,Automatic,2000,28524,2023-11-23,2025-06-24,Second,20782,6,3,13.622204,New,New,Weak,1
1,Van,60353,Average,1,7,Electric,Automatic,2500,133630,2023-09-21,2025-06-04,Second,23489,7,0,13.625307,New,New,Weak,1
2,Bus,68072,Poor,0,2,Electric,Automatic,1500,34022,2023-06-27,2025-04-27,First,17979,7,0,14.306302,New,Good,Weak,1
3,Bus,60849,Average,4,5,Petrol,Automatic,2500,81636,2023-08-24,2025-11-05,Second,6220,7,3,18.709467,New,Worn Out,New,1
4,Bus,45742,Poor,5,1,Petrol,Manual,2000,97162,2023-05-25,2025-09-14,Third,16446,6,2,16.977482,Good,Good,Weak,1


## 3. Filter Vehicle Model

Data vehicle difilter hanya untuk `Car` dan `Truck` karena sesuai kebutuhan data kendaraan.

In [3]:
vehicle_car_truck = vehicle_data[vehicle_data["Vehicle_Model"].isin(["Car", "Truck"])].copy()

vehicle_tidak_maintenance = vehicle_car_truck[vehicle_car_truck["Need_Maintenance"] == 0].copy()
vehicle_perlu_maintenance = vehicle_car_truck[vehicle_car_truck["Need_Maintenance"] == 1].copy()

print(f"Vehicle Car/Truck: {len(vehicle_car_truck)} baris")
print(f"Need_Maintenance=0: {len(vehicle_tidak_maintenance)} baris")
print(f"Need_Maintenance=1: {len(vehicle_perlu_maintenance)} baris")
display(pd.crosstab(vehicle_car_truck["Vehicle_Model"], vehicle_car_truck["Need_Maintenance"]))

Vehicle Car/Truck: 16531 baris
Need_Maintenance=0: 3050 baris
Need_Maintenance=1: 13481 baris


Need_Maintenance,0,1
Vehicle_Model,,
Car,1496,6707
Truck,1554,6774


## 4. Hitung Komponen KM yang Memenuhi Kondisi Servis

Komponen KM yang dicek adalah `km_oli`, `km_rem`, `km_busi`, dan `km_ban`. Mengikuti contoh aturan, sebuah komponen dianggap memenuhi kondisi servis jika nilainya `<= 10000`.

In [4]:
kolom_km = ["km_oli", "km_rem", "km_busi", "km_ban"]
komponen_km = {
    "km_oli": "oli",
    "km_rem": "rem",
    "km_busi": "busi",
    "km_ban": "ban",
}

def daftar_komponen_km_servis(row):
    return [nama for kolom, nama in komponen_km.items() if row[kolom] <= BATAS_KM_SERVIS]

def jumlah_komponen_km_servis(row):
    return len(daftar_komponen_km_servis(row))

data_historis = data_historis.copy()
data_historis["jumlah_komponen_km_servis"] = data_historis.apply(jumlah_komponen_km_servis, axis=1)

print("Distribusi jumlah komponen KM yang memenuhi kondisi servis:")
display(data_historis["jumlah_komponen_km_servis"].value_counts().sort_index())
display(pd.crosstab(data_historis["kelas"], data_historis["jumlah_komponen_km_servis"]))

Distribusi jumlah komponen KM yang memenuhi kondisi servis:


jumlah_komponen_km_servis
1    85
2    94
3    21
Name: count, dtype: int64

jumlah_komponen_km_servis,1,2,3
kelas,,,
Layak,27,36,12
Perlu Servis,58,58,9


## 5. Aturan Label Augmentasi

Aturan label akhir:

- Jika `jumlah_komponen_km_servis >= 2` dan `Need_Maintenance = 1`, maka label akhir menjadi `Perlu Servis`.
- Selain kondisi tersebut, label akhir menjadi `Layak`.

Dengan aturan ini, jika dua komponen historis memenuhi kondisi servis tetapi label vehicle adalah `Need_Maintenance = 0`, maka data tetap `Layak`.

In [5]:
def tentukan_label_akhir(row):
    if row["jumlah_komponen_km_servis"] >= 2 and row["Need_Maintenance"] == 1:
        return "Perlu Servis"
    return "Layak"

## 6. Siapkan Sampel Layak

Data `Layak` dibuat dari dua kombinasi:

- Vehicle `Need_Maintenance = 0` dengan data historis apa pun.
- Vehicle `Need_Maintenance = 1` tetapi data historis hanya memiliki kurang dari dua komponen KM yang memenuhi kondisi servis.

In [6]:
jumlah_layak_dari_vehicle_layak = 2000
jumlah_layak_dari_vehicle_perlu = TARGET_LAYAK - jumlah_layak_dari_vehicle_layak

historis_kurang_dari_dua = data_historis[data_historis["jumlah_komponen_km_servis"] < 2].copy()

if len(vehicle_tidak_maintenance) < jumlah_layak_dari_vehicle_layak:
    raise ValueError("Data vehicle Need_Maintenance=0 tidak cukup untuk sampel Layak.")
if len(vehicle_perlu_maintenance) < jumlah_layak_dari_vehicle_perlu:
    raise ValueError("Data vehicle Need_Maintenance=1 tidak cukup untuk sampel Layak edge case.")
if len(historis_kurang_dari_dua) == 0:
    raise ValueError("Tidak ada data historis dengan jumlah komponen KM servis kurang dari dua.")

layak_hist_a = data_historis.sample(n=jumlah_layak_dari_vehicle_layak, replace=True, random_state=RANDOM_STATE).reset_index(drop=True)
layak_vehicle_a = vehicle_tidak_maintenance.sample(n=jumlah_layak_dari_vehicle_layak, replace=False, random_state=RANDOM_STATE).reset_index(drop=True)

layak_hist_b = historis_kurang_dari_dua.sample(n=jumlah_layak_dari_vehicle_perlu, replace=True, random_state=RANDOM_STATE + 1).reset_index(drop=True)
layak_vehicle_b = vehicle_perlu_maintenance.sample(n=jumlah_layak_dari_vehicle_perlu, replace=False, random_state=RANDOM_STATE + 1).reset_index(drop=True)

print(f"Layak dari vehicle tidak maintenance: {len(layak_hist_a)}")
print(f"Layak dari vehicle perlu maintenance tetapi komponen historis < 2: {len(layak_hist_b)}")

Layak dari vehicle tidak maintenance: 2000
Layak dari vehicle perlu maintenance tetapi komponen historis < 2: 1000


## 7. Siapkan Sampel Perlu Servis

Data `Perlu Servis` hanya dibuat dari data historis dengan minimal dua komponen KM yang memenuhi kondisi servis dan vehicle `Need_Maintenance = 1`.

In [7]:
historis_minimal_dua = data_historis[data_historis["jumlah_komponen_km_servis"] >= 2].copy()

if len(vehicle_perlu_maintenance) < TARGET_PERLU_SERVIS + jumlah_layak_dari_vehicle_perlu:
    raise ValueError("Data vehicle Need_Maintenance=1 tidak cukup untuk semua sampel yang dibutuhkan.")
if len(historis_minimal_dua) == 0:
    raise ValueError("Tidak ada data historis dengan minimal dua komponen KM servis.")

perlu_hist = historis_minimal_dua.sample(n=TARGET_PERLU_SERVIS, replace=True, random_state=RANDOM_STATE + 2).reset_index(drop=True)
vehicle_perlu_sisa = vehicle_perlu_maintenance.drop(layak_vehicle_b.index, errors="ignore")
perlu_vehicle = vehicle_perlu_maintenance.sample(n=TARGET_PERLU_SERVIS, replace=False, random_state=RANDOM_STATE + 2).reset_index(drop=True)

print(f"Perlu Servis: {len(perlu_hist)}")

Perlu Servis: 10000


## 8. Seleksi, Rename, dan Konversi Nilai Vehicle

In [8]:
atribut_dipakai = [
    "Vehicle_Model",
    "Maintenance_History",
    "Reported_Issues",
    "Accident_History",
    "Brake_Condition",
    "Tire_Condition",
    "Battery_Status",
    "Need_Maintenance",
]

rename_atribut = {
    "Vehicle_Model": "jenis_kendaraan",
    "Maintenance_History": "riwayat_perawatan",
    "Reported_Issues": "jumlah_keluhan",
    "Accident_History": "riwayat_kecelakaan",
    "Brake_Condition": "kondisi_rem",
    "Tire_Condition": "kondisi_ban",
    "Battery_Status": "kondisi_aki",
}

mapping_nilai = {
    "jenis_kendaraan": {"Car": "mobil", "Truck": "truk"},
    "riwayat_perawatan": {"Good": "Baik", "Average": "Cukup", "Poor": "Buruk"},
    "kondisi_rem": {"New": "Baru", "Good": "Baik", "Worn Out": "Aus"},
    "kondisi_ban": {"New": "Baru", "Good": "Baik", "Worn Out": "Aus"},
    "kondisi_aki": {"New": "Baru", "Good": "Baik", "Weak": "Lemah"},
}

def siapkan_vehicle(dataframe):
    hasil = dataframe[atribut_dipakai].rename(columns=rename_atribut).copy()
    for kolom, peta in mapping_nilai.items():
        hasil[kolom] = hasil[kolom].replace(peta)
    return hasil

layak_vehicle_a = siapkan_vehicle(layak_vehicle_a)
layak_vehicle_b = siapkan_vehicle(layak_vehicle_b)
perlu_vehicle = siapkan_vehicle(perlu_vehicle)

## 9. Gabungkan Data Historis dan Vehicle

In [9]:
kolom_vehicle_baru = [
    "jenis_kendaraan",
    "riwayat_perawatan",
    "jumlah_keluhan",
    "riwayat_kecelakaan",
    "kondisi_rem",
    "kondisi_ban",
    "kondisi_aki",
    "Need_Maintenance",
]

layak_a = pd.concat([layak_hist_a.reset_index(drop=True), layak_vehicle_a[kolom_vehicle_baru].reset_index(drop=True)], axis=1)
layak_b = pd.concat([layak_hist_b.reset_index(drop=True), layak_vehicle_b[kolom_vehicle_baru].reset_index(drop=True)], axis=1)
perlu = pd.concat([perlu_hist.reset_index(drop=True), perlu_vehicle[kolom_vehicle_baru].reset_index(drop=True)], axis=1)

data_historis_gabungan = pd.concat([layak_a, layak_b, perlu], ignore_index=True)
data_historis_gabungan["kelas"] = data_historis_gabungan.apply(tentukan_label_akhir, axis=1)
data_historis_gabungan = data_historis_gabungan.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

display(data_historis_gabungan.head())
display(data_historis_gabungan["kelas"].value_counts())

,id,umur_kendaraan,jarak_tempuh_tahun,frekuensi_servis_tahun,km_oli,km_rem,km_busi,km_ban,kelas,created_at,updated_at,jumlah_komponen_km_servis,jenis_kendaraan,riwayat_perawatan,jumlah_keluhan,riwayat_kecelakaan,kondisi_rem,kondisi_ban,kondisi_aki,Need_Maintenance
0,853,1,25040,5,3976,14112,9616,23928,Perlu Servis,2026-07-02 02:26:06,2026-07-02 02:26:06,2,mobil,Buruk,4,3,Baru,Baik,Baik,1
1,1008,5,25305,5,3765,30295,11505,23165,Layak,2026-07-02 02:26:06,2026-07-02 02:26:06,1,truk,Buruk,0,1,Aus,Baru,Baik,1
2,943,6,27706,4,4238,12998,8946,77626,Perlu Servis,2026-07-02 02:26:06,2026-07-02 02:26:06,2,mobil,Buruk,2,3,Baru,Aus,Baru,1
3,906,2,12887,3,1651,21153,6367,11811,Perlu Servis,2026-07-02 02:26:06,2026-07-02 02:26:06,2,truk,Buruk,2,2,Aus,Aus,Baru,1
4,831,3,8980,3,4562,16894,14542,31086,Layak,2026-07-02 02:26:05,2026-07-02 02:26:05,1,truk,Buruk,1,2,Baik,Baik,Baik,0


kelas
Perlu Servis    10000
Layak            3000
Name: count, dtype: int64

## 10. Tambahkan Kolom Diagnosis untuk Audit

Kolom diagnosis ini membantu mengecek aturan. Pada tahap modelling, kolom diagnosis tidak digunakan sebagai fitur agar tidak terjadi kebocoran data.

In [10]:
tingkat_servis = {
    "Servis 10.000 KM": 1,
    "Servis 20.000 KM": 2,
    "Servis 40.000 KM": 3,
    "Servis 60.000 KM": 4,
    "Servis Besar": 5,
}

def tentukan_jenis_servis(nilai_km):
    if nilai_km <= 10000:
        return "Servis 10.000 KM"
    if nilai_km <= 20000:
        return "Servis 20.000 KM"
    if nilai_km <= 40000:
        return "Servis 40.000 KM"
    if nilai_km <= 60000:
        return "Servis 60.000 KM"
    return "Servis Besar"

kolom_km_servis = {
    "km_oli": "jenis_servis_oli",
    "km_rem": "jenis_servis_rem",
    "km_busi": "jenis_servis_busi",
    "km_ban": "jenis_servis_ban",
}

for kolom_km_item, kolom_servis in kolom_km_servis.items():
    data_historis_gabungan[kolom_servis] = data_historis_gabungan[kolom_km_item].apply(tentukan_jenis_servis)

def komponen_perlu_servis(row):
    komponen = daftar_komponen_km_servis(row)
    if row["kondisi_rem"] == "Aus" and "rem" not in komponen:
        komponen.append("rem")
    if row["kondisi_ban"] == "Aus" and "ban" not in komponen:
        komponen.append("ban")
    if row["kondisi_aki"] == "Lemah" and "aki" not in komponen:
        komponen.append("aki")
    return ", ".join(komponen) if komponen else "Belum ada komponen prioritas"

def jenis_servis_tertinggi(row):
    daftar_servis = [row[kolom_servis] for kolom_servis in kolom_km_servis.values()]
    return max(daftar_servis, key=lambda nilai: tingkat_servis[nilai])

data_historis_gabungan["jenis_servis_tertinggi"] = data_historis_gabungan.apply(jenis_servis_tertinggi, axis=1)
data_historis_gabungan["komponen_perlu_servis"] = data_historis_gabungan.apply(komponen_perlu_servis, axis=1)

display(data_historis_gabungan[["jumlah_komponen_km_servis", "Need_Maintenance", "kelas", "komponen_perlu_servis"]].head())

,jumlah_komponen_km_servis,Need_Maintenance,kelas,komponen_perlu_servis
0,2,1,Perlu Servis,"oli, busi"
1,1,1,Layak,"oli, rem"
2,2,1,Perlu Servis,"oli, busi, ban"
3,2,1,Perlu Servis,"oli, busi, rem, ban"
4,1,0,Layak,oli


## 11. Verifikasi Aturan Label

In [11]:
assert len(data_historis_gabungan) == TARGET_LAYAK + TARGET_PERLU_SERVIS
assert data_historis_gabungan["kelas"].value_counts().to_dict() == {
    "Perlu Servis": TARGET_PERLU_SERVIS,
    "Layak": TARGET_LAYAK,
}

cek_label = data_historis_gabungan.apply(tentukan_label_akhir, axis=1)
assert (cek_label == data_historis_gabungan["kelas"]).all(), "Masih ada label yang tidak sesuai aturan."

perlu_valid = data_historis_gabungan[data_historis_gabungan["kelas"] == "Perlu Servis"]
assert (perlu_valid["jumlah_komponen_km_servis"] >= 2).all()
assert (perlu_valid["Need_Maintenance"] == 1).all()

print("Verifikasi berhasil.")
print(f"Total data: {len(data_historis_gabungan)}")
display(data_historis_gabungan["kelas"].value_counts())
display(pd.crosstab(data_historis_gabungan["Need_Maintenance"], data_historis_gabungan["kelas"]))
display(pd.crosstab(data_historis_gabungan["jumlah_komponen_km_servis"], data_historis_gabungan["kelas"]))

Verifikasi berhasil.
Total data: 13000


kelas
Perlu Servis    10000
Layak            3000
Name: count, dtype: int64

kelas,Layak,Perlu Servis
Need_Maintenance,,
0,2000,0
1,1000,10000


kelas,Layak,Perlu Servis
jumlah_komponen_km_servis,,
1,1856,0
2,933,8154
3,211,1846


## 12. Simpan Dataset Gabungan

Kolom `Need_Maintenance` disimpan sebagai kolom audit sumber label vehicle. Pada modelling, kolom ini harus dikeluarkan dari fitur karena berpotensi menjadi kebocoran label.

In [12]:
data_historis_gabungan.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset gabungan berhasil disimpan ke: {OUTPUT_PATH}")

Dataset gabungan berhasil disimpan ke: data_historis_gabungan.csv
